# 03 — Modeling and Evaluation

Goals:
- Establish a majority-class baseline
- Train QDA and Gaussian Naive Bayes
- Use the finalized pre-contact feature set
- Use consistent train/validation/test splits
- Compare classification metrics
- Add stratified cross-validation

## Load the Data

In [ ]:
import sys
from pathlib import Path

import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

sys.path.append(str(PROJECT_ROOT))

from src.evaluation import (
    evaluate_binary_classifier,
    get_confusion_matrix,
    results_to_dataframe,
    plot_confusion_matrix,
    plot_roc_curve,
    cross_validate_classifier,
    cross_validation_results_to_dataframe,
)

from src.preprocessing import build_preprocessor
from src.models import get_student3_models

DATA_PATH = PROJECT_ROOT / "data" / "bank-full.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Download bank-full.csv and place it in the data/ folder."
    )

df = pd.read_csv(DATA_PATH, sep=";")

df.head()

In [ ]:
X = df.drop(columns=["y"])

y = df["y"].map({
    "no": 0,
    "yes": 1,
})

## Finalized Pre-Contact Feature Set

Following the EDA and leakage analysis, the primary modeling workflow
uses 15 predictors that are available at the stated pre-contact
prediction point.

`duration` is excluded because the length of the current call is not
known before the call occurs.

In [ ]:
numeric_features = [
    "age",
    "balance",
    "day",
    "campaign",
    "pdays",
    "previous",
]

categorical_features = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "poutcome",
]

model_features = numeric_features + categorical_features

X_model = X[model_features]

print("Number of modeling features:", len(model_features))
print(model_features)

## Train / Validation / Test Split

The finalized modeling dataset is split using stratified sampling into
70% training, 15% validation, and 15% testing data.

Using the same split strategy and random state as the other candidate
models allows final performance results to be compared on the same
observations.

In [ ]:
X_train_model, X_temp_model, y_train_model, y_temp_model = train_test_split(
    X_model,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

X_test_model, X_val_model, y_test_model, y_val_model = train_test_split(
    X_temp_model,
    y_temp_model,
    test_size=0.50,
    stratify=y_temp_model,
    random_state=42,
)

print("Training:", X_train_model.shape)
print("Validation:", X_val_model.shape)
print("Testing:", X_test_model.shape)

print("Training positive rate:", y_train_model.mean())
print("Validation positive rate:", y_val_model.mean())
print("Testing positive rate:", y_test_model.mean())

## Majority-Class Baseline

Because the target is imbalanced, the baseline predicts the most
frequent class for every observation. Candidate models should be
compared against this baseline rather than using accuracy alone.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train_model, y_train_model)

baseline_predictions = baseline.predict(X_test_model)
baseline_probabilities = baseline.predict_proba(X_test_model)[:, 1]

baseline_results = evaluate_binary_classifier(
    model_name="Majority Class Baseline",
    y_true=y_test_model,
    y_pred=baseline_predictions,
    y_score=baseline_probabilities,
)

baseline_results

In [ ]:
baseline_cm = get_confusion_matrix(
    y_test_model,
    baseline_predictions,
)

baseline_cm

In [ ]:
plot_confusion_matrix(
    y_true=y_test_model,
    y_pred=baseline_predictions,
    model_name="Majority Class Baseline",
)

In [ ]:
plot_roc_curve(
    y_true=y_test_model,
    y_score=baseline_probabilities,
    model_name="Majority Class Baseline",
)

In [ ]:
baseline_cv_results = cross_validate_classifier(
    model_name="Majority Class Baseline",
    estimator=baseline,
    X=X_train_model,
    y=y_train_model,
    cv=5,
)

baseline_cv_results

## QDA and Gaussian Naive Bayes

QDA and Gaussian Naive Bayes are evaluated using the finalized
15-feature pre-contact modeling dataset defined above.

Both models require dense transformed input.

QDA additionally drops the first one-hot encoded category from each
categorical predictor to reduce dummy-variable collinearity.

In [ ]:
student3_models = get_student3_models()

### Quadratic Discriminant Analysis

In [ ]:
qda_model = student3_models[
    "Quadratic Discriminant Analysis"
]

qda_preprocessor = build_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    dense_output=True,
    drop_first=True,
)

qda_pipeline = Pipeline(
    steps=[
        ("preprocessor", qda_preprocessor),
        ("classifier", qda_model),
    ]
)

qda_pipeline.fit(
    X_train_model,
    y_train_model,
)

QDA previously encountered rank-deficiency problems because full
one-hot encoding produced collinear predictors.

The current QDA workflow drops the first category from each categorical
predictor and uses `reg_param=0.1` to improve covariance-estimation
stability.

In [ ]:
qda_predictions = qda_pipeline.predict(X_test_model)
qda_probabilities = qda_pipeline.predict_proba(X_test_model)[:, 1]

qda_results = evaluate_binary_classifier(
    model_name="Quadratic Discriminant Analysis",
    y_true=y_test_model,
    y_pred=qda_predictions,
    y_score=qda_probabilities,
)

qda_results

In [ ]:
plot_confusion_matrix(
    y_true=y_test_model,
    y_pred=qda_predictions,
    model_name="Quadratic Discriminant Analysis",
)

In [ ]:
plot_roc_curve(
    y_true=y_test_model,
    y_score=qda_probabilities,
    model_name="Quadratic Discriminant Analysis",
)

In [ ]:
qda_cv_results = cross_validate_classifier(
    model_name="Quadratic Discriminant Analysis",
    estimator=qda_pipeline,
    X=X_train_model,
    y=y_train_model,
    cv=5,
)

qda_cv_results

### Gaussian Naive Bayes

In [ ]:
gnb_preprocessor = build_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    dense_output=True,
)

gnb_model = student3_models[
    "Gaussian Naive Bayes"
]

gnb_pipeline = Pipeline(
    steps=[
        ("preprocessor", gnb_preprocessor),
        ("classifier", gnb_model),
    ]
)

gnb_pipeline.fit(
    X_train_model,
    y_train_model,
)

In [ ]:
gnb_predictions = gnb_pipeline.predict(X_test_model)
gnb_probabilities = gnb_pipeline.predict_proba(X_test_model)[:, 1]

gnb_results = evaluate_binary_classifier(
    model_name="Gaussian Naive Bayes",
    y_true=y_test_model,
    y_pred=gnb_predictions,
    y_score=gnb_probabilities,
)

gnb_results

In [ ]:
plot_confusion_matrix(
    y_true=y_test_model,
    y_pred=gnb_predictions,
    model_name="Gaussian Naive Bayes",
)

In [ ]:
plot_roc_curve(
    y_true=y_test_model,
    y_score=gnb_probabilities,
    model_name="Gaussian Naive Bayes",
)

In [ ]:
gnb_cv_results = cross_validate_classifier(
    model_name="Gaussian Naive Bayes",
    estimator=gnb_pipeline,
    X=X_train_model,
    y=y_train_model,
    cv=5,
)

gnb_cv_results

## Student 3 Test-Set Comparison

In [ ]:
results = [
    baseline_results,
    qda_results,
    gnb_results,
]

results_df = results_to_dataframe(results)

results_df

## Student 3 Cross-Validation Comparison

In [ ]:
cv_results = [
    baseline_cv_results,
    qda_cv_results,
    gnb_cv_results,
]

cv_results_df = cross_validation_results_to_dataframe(
    cv_results
)

cv_results_df

## Effect of the Finalized Feature Set

The original Student 3 experiments used 11 predictors and excluded
`default`, `day`, `duration`, `pdays`, and `poutcome`.

Following the finalized EDA and leakage analysis, the primary
pre-contact feature set now contains 15 predictors. `duration` remains
excluded because it is unavailable before the current call, while
`default`, `day`, `pdays`, and `poutcome` are retained as information
that can be available at the stated prediction time.

The updated QDA and Gaussian Naive Bayes results should be compared
with the earlier experiments using recall, F1-score, ROC-AUC, and
cross-validation stability in addition to accuracy.